# Model Golf · Tiny ECG challenge
MassAI · November 18, 2026

Your team’s goal is to submit the fewest model bytes while meeting both scoring floors. The model classifies pre-cut heartbeats as `N` (normal beat) or `V` (premature ventricular contraction, PVC). Each example comes from the MIT-BIH annotation at that beat. Other beat labels are excluded.

**Start here:** Open this notebook in [Colab](https://colab.research.google.com/) via **File → Upload notebook**. Use the CPU runtime. Run the setup cell and upload `model-golf-data.zip` from the student kit when prompted. For local Jupyter, keep the data ZIP beside this notebook.

Work in teams of 2–3. Have one person run trials and one person record the results. Switch roles halfway through.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys

archive_name = 'model-golf-data.zip'
base = Path('.')
if not (base / 'classroom.npz').exists():
    if not (base / archive_name).exists():
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError(f'Put {archive_name} beside this notebook.') from error
        print(f'Choose {archive_name} in the upload dialog.')
        uploaded = files.upload()
        if archive_name not in uploaded:
            raise ValueError(f'Please upload the supplied {archive_name}.')
        (base / archive_name).write_bytes(uploaded[archive_name])
    base = Path('model-golf-data')
    members = ('classroom.npz', 'demo_cases.npz', 'baseline.mgf', 'model_golf.py',
               'rules.json', 'wokwi/diagram.json', 'wokwi/src/main.cpp', 'wokwi/src/inference.h')
    with ZipFile(archive_name) as archive:
        for name in members:
            target = base / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))

import numpy as np
sys.path.insert(0, str(base.resolve()))
from model_golf import (LABELS, WIDTHS, browser_sketch, decode, encode, firmware_headers,
                        format_score, logits, score, train)

with np.load(base / 'classroom.npz', allow_pickle=False) as data:
    x_train, y_train = data['train_x'], data['train_y']
    x_val, y_val = data['validation_x'], data['validation_y']
with np.load(base / 'demo_cases.npz', allow_pickle=False) as data:
    cases = {name: data[name] for name in ('id', 'x', 'y')}
baseline = (base / 'baseline.mgf').read_bytes()
print(f'{len(x_train):,} training windows · {len(x_val):,} practice windows · {x_train.shape[1]} values per window')

## 1 · Run the baseline

Accuracy is the fraction of all beats classified correctly. PVC sensitivity is the fraction of annotated PVC beats caught. Think about what the always-N model misses.

In [ ]:
baseline_score = score(baseline, x_val, y_val)
print('Pretrained 24-neuron FP32 model')
print(format_score(baseline_score))
print(f'Always-N practice accuracy: {np.mean(y_val == 0):.1%} | PVC sensitivity: 0.0%')

## 2 · Change two settings

- `width` controls the number of hidden neurons. Choose 4, 8, 16, or 24. A smaller width retrains a narrower model on the supplied training partition.
- `bits` controls weight storage. Choose 32 (FP32) or 8 (INT8). INT8 rounds the weights and stores a scale for each layer. Biases and inference arithmetic use FP32 in both modes.

Run several trials. Start by keeping `width = 24` and changing `bits` to 8 to isolate quantization. Then shrink the width. Compare the measured bytes, accuracy and sensitivity together.

In [ ]:
import json

rules = json.loads((base / 'rules.json').read_text())
accuracy_floor = rules['minimum_accuracy']
sensitivity_floor = rules['minimum_pvc_sensitivity']
trials = {(24, 32): baseline_score}
fitted = {24: decode(baseline)}
payloads = {(24, 32): baseline}
print(f'Final floors: {accuracy_floor:.0%} accuracy and {sensitivity_floor:.0%} PVC sensitivity.')

In [ ]:
# Edit these two numbers and rerun this cell.
width = 24
bits = 8

if width not in WIDTHS or bits not in (8, 32):
    raise ValueError('Use a width of 4, 8, 16, or 24 and bits of 8 or 32.')
if width not in fitted:
    fitted[width] = train(x_train, y_train, width=width)
payload = encode(fitted[width], bits=bits)
payloads[width, bits] = payload
trials[width, bits] = score(payload, x_val, y_val)
for (w, b), result in sorted(trials.items(), key=lambda pair: pair[1]['bytes']):
    print(f'Width {w:2d} / {b:2d}-bit  ·  {format_score(result)}')

## 3 · Send one model to the facilitator

Choose a trial with room above both floors. Practice scores are estimates. The facilitator will rank the uploaded model files using different recordings and a shared evaluation script. Among qualified submissions, the fewest bytes wins. Higher PVC sensitivity breaks a tie, followed by higher accuracy. An exact tie shares the rank.

Your score uses the binary payload size in bytes. It includes the model header, weights, biases and any quantization scales. Name one `.mgf` file for your team and hand it to the facilitator through the workshop’s collection channel.

In [ ]:
import re

team = 'YourTeam'
submit_width = width
submit_bits = bits
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9 _-]{0,39}', team) or team == 'YourTeam':
    print('Set team to your actual 1–40 character team name and rerun this cell.')
else:
    pick = (submit_width, submit_bits)
    if pick not in payloads:
        raise ValueError('Run that trial before choosing it.')
    submission = Path(team + '.mgf')
    submission.write_bytes(payloads[pick])
    print(f'Ready: {submission} ({submission.stat().st_size} bytes)')
    try:
        from google.colab import files
    except ImportError:
        print(submission.resolve())
    else:
        files.download(str(submission))

## Debrief

1. How many model bytes did your change save? Which metric moved most?
2. Which error does PVC sensitivity count? What other error should we inspect?
3. Why could a model do well on the practice recordings and struggle on a new recording?

[MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/) · [WFDB beat annotation codes](https://physionet.org/physiotools/wpg/wpg_36.htm)